If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec01_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 1: Plotting the Classics
v.ekc

Welcome to DATA 111! 😊 Today isn't about memorizing code — it's a preview of what data science *feels* like: we'll take two whole novels and turn them into pictures that tell a story. (Slides: KL Lecture 01 deck, slides 36–41.)

**Today's sections:**
1. Getting set up
2. Read two books, fast!
3. Counting character mentions — *Huckleberry Finn*
4. Same idea, new book — *Little Women*
5. How long is a chapter?
6. Try It

---
## 1. Getting set up

Every notebook starts by importing the tools we'll use — most importantly the `datascience` package ([docs](https://data8.org/datascience/)), built for this course. Run this cell and move on; you'll get to know these lines well this semester.

In [ ]:
from datascience import *
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

from urllib.request import urlopen 
import re
def read_url(url): 
    return re.sub('\\s+', ' ', urlopen(url).read().decode())

And just to prove the notebook is alive — Python is, at minimum, a very fancy calculator:

In [ ]:
2+3

---
## 2. Read two books, fast! 📚

*The Adventures of Huckleberry Finn* and *Little Women* are in the public domain, so we can pull the full texts straight off the web and split them into chapters. Don't worry about *how* each line works yet — notice how **little** code it takes.

In [ ]:
# Read two books, fast!

huck_finn_url = 'https://www.inferentialthinking.com/data/huck_finn.txt'
huck_finn_text = read_url(huck_finn_url)
huck_finn_chapters = huck_finn_text.split('CHAPTER ')[44:]

little_women_url = 'https://www.inferentialthinking.com/data/little_women.txt'
little_women_text = read_url(little_women_url)
little_women_chapters = little_women_text.split('CHAPTER ')[1:]

In [ ]:
huck_finn_chapters

In [ ]:
Table().with_column('Chapters', huck_finn_chapters)

> **Key idea:** a **Table** is how we'll organize data all semester — rows and columns, like a spreadsheet with superpowers.

---
## 3. Counting character mentions — *Huckleberry Finn*

If we count how many times each character's name appears in each chapter, we can watch the story unfold — without reading a single page. 🔍

### 📋 Board Reference

| Code | What it does |
|---|---|
| `Table()` | makes an empty table |
| `.with_columns('label', array, ...)` | returns a table with new columns added |
| `.column('label')` | grabs one column as an array |
| `np.char.count(chapters, 'Tom')` | counts `'Tom'` in each chapter |
| `.cumsum()` | running total of an array |
| `.plot(...)` | line plot, one line per column |

In [ ]:
np.char.count(huck_finn_chapters, 'Tom')

In [ ]:
np.char.count(huck_finn_chapters, 'Jim')

In [ ]:
counts = Table().with_columns([
    'Tom', np.char.count(huck_finn_chapters, 'Tom'),
    'Jim', np.char.count(huck_finn_chapters, 'Jim'),
    'Huck', np.char.count(huck_finn_chapters, 'Huck'),
])
counts

In [ ]:
# Plot the cumulative counts:
# how many times in Chapter 1, how many times in Chapters 1 and 2, and so on.

cum_counts = Table().with_columns([
    'Tom', counts.column('Tom').cumsum(),
    'Jim', counts.column('Jim').cumsum(),
    'Huck', counts.column('Huck').cumsum(),
]).with_column('Chapter', np.arange(1, 44, 1))
cum_counts.plot(column_for_xticks=3)
plots.title('Cumulative Number of Times Name Appears');

> **Reading the plot:** Tom's line goes flat through the middle of the book — he literally disappears from the story — then climbs when he returns at the end. Jim's steady rise tells you he's with Huck for the whole journey. We just "read" a novel's plot structure from a graph. 🤯

---
## 4. Same idea, new book — *Little Women*

The same trick works on any book. This time: the four March sisters, plus their neighbor Laurie.

In [ ]:
# The chapters of Little Women

Table().with_column('Chapters', little_women_chapters)

In [ ]:
# Counts of names in the chapters of Little Women

names = ['Amy', 'Beth', 'Jo', 'Laurie', 'Meg']
mentions = {name: np.char.count(little_women_chapters, name) for name in names}

counts = Table().with_columns([
        'Amy', mentions['Amy'],
        'Beth', mentions['Beth'],
        'Jo', mentions['Jo'],
        'Laurie', mentions['Laurie'],
        'Meg', mentions['Meg']
    ])

In [ ]:
# Plot the cumulative counts

cum_counts = Table().with_columns([
    'Amy', counts.column('Amy').cumsum(),
    'Beth', counts.column('Beth').cumsum(),
    'Jo', counts.column('Jo').cumsum(),
    'Laurie', counts.column('Laurie').cumsum(),
    'Meg', counts.column('Meg').cumsum(),
]).with_column('Chapter', np.arange(1, 48, 1))
cum_counts.plot(column_for_xticks=5)
plots.title('Cumulative Number of Times Name Appears');

> **Try reading this one yourself:** Who seems to be the main character? What might a line going *flat* partway through mean? 💭 You'll work with this exact plot on **Homework 1**, so it's worth lingering here.

---
## 5. How long is a chapter?

Names aren't the only thing we can count. Let's measure each chapter two ways — total characters (letters, spaces, everything) and number of periods — and compare the two books.

In [ ]:
len('Data 111')

In [ ]:
len(read_url(huck_finn_url))

In [ ]:
# In each chapter, count the number of all characters;
# call this the "length" of the chapter.
# Also count the number of periods.

length_hf = Table().with_columns([
        'Length', [len(s) for s in huck_finn_chapters],
        'Periods', np.char.count(huck_finn_chapters, '.')
    ])
length_lw = Table().with_columns([
        'Length', [len(s) for s in little_women_chapters],
        'Periods', np.char.count(little_women_chapters, '.')
    ])

In [ ]:
# The counts for Huckleberry Finn

length_hf

In [ ]:
# The counts for Little Women

length_lw

In [ ]:
plots.figure(figsize=(10,10))
plots.scatter(length_hf[1], length_hf[0], color='darkblue')
plots.scatter(length_lw[1], length_lw[0], color='gold')
plots.xlabel('Number of periods in chapter')
plots.ylabel('Number of characters in chapter');

> **One dot = one chapter.** Both books show the same trend: chapters with more periods are longer — more sentences, more text. When two measurements move together like this, we call it **association**, a big idea we'll return to all semester.

---
### Try It 1: Your turn with the classics 😊

1. Count how many times `'river'` appears in each chapter of *Huckleberry Finn*.

2. Make a cumulative plot of the `'river'` counts, chapter by chapter. Does the river matter more in some parts of the book?

3. Interpretation: look back at the *Little Women* plot. Which character is mentioned most by the end of the book, and roughly where does the gap open up?

In [ ]:
# 1. count 'river' in each chapter


In [ ]:
# 2. cumulative plot of 'river'


*3. Your interpretation: (double-click to edit)*

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*The same three-step pattern — count, accumulate, plot — works for any word you're curious about.*

```python
# 1.
np.char.count(huck_finn_chapters, 'river')

# 2.
river = Table().with_columns(
    'River', np.char.count(huck_finn_chapters, 'river').cumsum(),
    'Chapter', np.arange(1, 44, 1))
river.plot('Chapter')
```

*3. Jo — her line is on top almost from the start, and the gap widens within the first dozen chapters. (Many right answers — what matters is pointing at evidence in the plot.)*

</details>

---
> **You made it through your first data science lecture!** 🎉 None of today's code is expected to stick yet — every piece gets a proper introduction in the coming weeks. What should stick: *we turned books into data, and data into stories.*

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/) • Free textbook: [Computational and Inferential Thinking](https://www.inferentialthinking.com)

### Minimal template — count & plot
```python
counts = Table().with_columns(
    'Word', np.char.count(chapters, 'word').cumsum(),
    'Chapter', np.arange(1, len(chapters) + 1))
counts.plot('Chapter')
```